# Phase 4 — Evaluation

Deep evaluation of the best model:
- Coverage: what % of the catalog ever gets recommended?
- Diversity per query
- Similarity score distribution
- Popularity bias check

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
import joblib
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
import seaborn as sns

MODELS_DIR = '../models'

df             = joblib.load(f'{MODELS_DIR}/perfume_df.pkl')
knn            = joblib.load(f'{MODELS_DIR}/knn_model.pkl')
mlb            = joblib.load(f'{MODELS_DIR}/mlb_ingredients.pkl')
ohe            = joblib.load(f'{MODELS_DIR}/ohe_categories.pkl')
tfidf          = joblib.load(f'{MODELS_DIR}/tfidf_description.pkl')
best_approach  = joblib.load(f'{MODELS_DIR}/best_approach.pkl')

matrix = load_npz(f'{MODELS_DIR}/matrix_{best_approach}.npz')

print(f'Best approach: {best_approach}')
print(f'Dataset size : {len(df):,}')

## 1. Catalog Coverage (1,000 random queries)

In [ ]:
np.random.seed(42)
sample_indices = np.random.choice(len(df), size=1000, replace=False)
sample_queries = matrix[sample_indices]

distances, indices = knn.kneighbors(sample_queries, n_neighbors=10)

all_recommended = set(indices.flatten().tolist())
coverage = len(all_recommended) / len(df)
print(f'Unique perfumes recommended: {len(all_recommended):,}')
print(f'Catalog coverage           : {coverage:.1%}')

## 2. Similarity Score Distribution

In [ ]:
# Skip distance[0] (self-match = 0 distance)
sim_scores = (1 - distances[:, 1:]).flatten()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sim_scores, bins=50, color='steelblue', edgecolor='white')
ax.axvline(np.mean(sim_scores), color='tomato', linestyle='--',
           label=f'Mean: {np.mean(sim_scores):.3f}')
ax.axvline(np.median(sim_scores), color='seagreen', linestyle='--',
           label=f'Median: {np.median(sim_scores):.3f}')
ax.set_title('Distribution of Similarity Scores (1,000 random queries)', fontweight='bold')
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../data/eval_similarity_dist.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mean similarity : {np.mean(sim_scores):.4f}')
print(f'Median similarity: {np.median(sim_scores):.4f}')
print(f'Min / Max       : {sim_scores.min():.4f} / {sim_scores.max():.4f}')

## 3. Popularity Bias — Are Some Perfumes Recommended Too Often?

In [ ]:
from collections import Counter

rec_counts = Counter(indices[:, 1:].flatten().tolist())
rec_series = pd.Series(rec_counts).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(rec_series)), rec_series.values)
ax.set_title('Recommendation Frequency per Perfume (Long-Tail)', fontweight='bold')
ax.set_xlabel('Perfume rank (sorted by recommendation count)')
ax.set_ylabel('Times recommended')
plt.tight_layout()
plt.savefig('../data/eval_popularity_bias.png', dpi=100, bbox_inches='tight')
plt.show()

# Top 10 most recommended
top_rec = df.iloc[rec_series.head(10).index][['name_perfume', 'brand', 'family']].copy()
top_rec['rec_count'] = rec_series.head(10).values
print('Top 10 most recommended perfumes:')
print(top_rec.to_string(index=False))

## 4. Diversity per Query

In [ ]:
from sklearn.metrics.pairwise import cosine_distances

ild_scores = []
for i in range(min(200, len(sample_indices))):
    idx = indices[i, 1:]  # skip self
    sub = matrix[idx]
    dists = cosine_distances(sub)
    n = len(idx)
    upper = dists[np.triu_indices(n, k=1)]
    ild_scores.append(upper.mean())

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ild_scores, bins=30, color='seagreen', edgecolor='white')
ax.axvline(np.mean(ild_scores), color='tomato', linestyle='--',
           label=f'Mean ILD: {np.mean(ild_scores):.3f}')
ax.set_title('Intra-List Diversity Distribution (200 random queries)', fontweight='bold')
ax.set_xlabel('Average Pairwise Cosine Distance within Top-10')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('../data/eval_ild.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Mean ILD: {np.mean(ild_scores):.4f}')

## 5. Evaluation Summary

In [ ]:
comparison_df = pd.read_csv(f'{MODELS_DIR}/model_comparison.csv')

print('=== EVALUATION SUMMARY ===')
print(f'Best approach          : {best_approach}')
print(f'Dataset size           : {len(df):,}')
print(f'Catalog coverage       : {coverage:.1%}')
print(f'Mean similarity score  : {np.mean(sim_scores):.4f}')
print(f'Mean intra-list div.   : {np.mean(ild_scores):.4f}')
print()
print('Model comparison:')
print(comparison_df.to_string(index=False))